# Project: Email assistant with safeguards

**Module:** Module 3 — Production-Oriented Agent Design  
**Notebook:** `notebooks/module-3/3.5_email_agent.ipynb`

## Purpose
Assemble an email workflow with authentication/state-aware capabilities, dynamic prompts/tools and human review before sending or changing email content.

## Workflow
1. **Prepare the environment** — load required packages and local environment variables. Keep API keys in an ignored `.env` file; never paste credentials into notebook cells.
2. **Build the capability** — configure the model/agent, messages, tools, middleware or data source used in this lesson.
3. **Execute the example** — invoke or stream the agent and inspect the returned messages, tool calls, structured response or state.
4. **Validate behavior** — compare the result with the intended behavior, test an edge case, and check provider/service availability if execution fails.
5. **Reflect** — use the key learnings below to explain the design choices and limitations.

## Learning objectives
- Understand the concept demonstrated by this notebook and where it fits in an agent workflow.
- Explain the role of the main LangChain/LangGraph abstractions used in the implementation.
- Trace inputs through execution and interpret the resulting output.
- Identify operational concerns such as credentials, external services, state, safety and error handling.

## Key learnings
- Long conversations need deliberate context management, not simply unlimited message history.
- Human-in-the-loop checkpoints are a control boundary for actions with external side effects.
- Dynamic model/prompt/tool selection can reduce cost and tailor capabilities, but adds complexity.

## Prerequisites and execution notes
- Use the repository setup instructions in the root `README.md` and the supported Python version.
- Configure only the provider credentials needed by this notebook in your local `.env` file.
- Some cells call external APIs, local MCP services, audio devices or sample databases; those dependencies must be available.
- Model output is non-deterministic and provider model names/API behavior can change. Treat outputs as examples, not fixed expected strings.

---



email agent
- authenticates user
    - only then are they allowed into the "inbox"
    - dynamic tools and prompt on the condition of there being an email and password in state that match hardcoded
- checks "inbox"
    - email in tool
- sends emails
    - human in the loop

### Implementation step 2

Imports the libraries and APIs needed by the following example: import warnings; from dotenv import load_dotenv. Loads local environment variables so provider/API credentials can be read without hard-coding secrets in the notebook.

**What to verify:** Check that the output matches the intended data flow, and inspect intermediate messages/tool calls when behavior is unclear.


In [ ]:
# Purpose: Imports the libraries and APIs needed by the following example: import warnings; from dotenv import load_dotenv. Loads local environment variables so provider/API credentials can be read without hard-coding secrets in the notebook.

import warnings
warnings.filterwarnings("ignore", message=".*Pydantic serializer warnings.*", category=UserWarning)

from dotenv import load_dotenv

load_dotenv()

### Implementation step 3

Imports the libraries and APIs needed by the following example: from dataclasses import dataclass.

**What to verify:** Check that the output matches the intended data flow, and inspect intermediate messages/tool calls when behavior is unclear.


In [ ]:
# Purpose: Imports the libraries and APIs needed by the following example: from dataclasses import dataclass.

from dataclasses import dataclass

@dataclass
class EmailContext:
    email_address: str = "julie@example.com"
    password: str = "password123"

### Implementation step 4

Imports the libraries and APIs needed by the following example: from langchain.agents import AgentState. Works with agent state, which carries structured information through the graph execution and may persist through a checkpoint.

**What to verify:** Check that the output matches the intended data flow, and inspect intermediate messages/tool calls when behavior is unclear.


In [ ]:
# Purpose: Imports the libraries and APIs needed by the following example: from langchain.agents import AgentState. Works with agent state, which carries structured information through the graph execution and may persist through a checkpoint.

from langchain.agents import AgentState

class AuthenticatedState(AgentState):
    authenticated: bool

### Implementation step 5

Imports the libraries and APIs needed by the following example: from langchain.tools import tool, ToolRuntime; from langgraph.types import Command; from langchain.messages import ToolMessage. Defines a reusable Python function/tool. The docstring and type annotations communicate the tool purpose and input schema to the agent. Uses runtime context to access request-scoped dependencies or contextual values without treating them as model-authored input. Demonstrates a pause/resume control path. Review the proposed action before resuming; approval should be explicit for consequential operations.

**What to verify:** Check that the output matches the intended data flow, and inspect intermediate messages/tool calls when behavior is unclear.


In [ ]:
# Purpose: Imports the libraries and APIs needed by the following example: from langchain.tools import tool, ToolRuntime; from langgraph.types import Command; from langchain.messages import ToolMessage. Defines a reusable Python function/tool. The docstring and type annotations communicate the tool purpose and input schema to the agent. Uses runtime context to access request-scoped dependencies or contextual values without treating them as model-authored input. Demonstrates a pause/resume control path. Review the proposed action before resuming; approval should be explicit for consequential operations.

from langchain.tools import tool, ToolRuntime
from langgraph.types import Command
from langchain.messages import ToolMessage

@tool
def check_inbox() -> str:
    """Check the inbox for recent emails"""
    return """
    Hi Julie, 
    I'm going to be in town next week and was wondering if we could grab a coffee?
    - best, Jane (jane@example.com)
    """

@tool
def send_email(to: str, subject: str, body: str) -> str:
    """Send an response email"""
    return f"Email sent to {to} with subject {subject} and body {body}"

@tool
def authenticate(email: str, password: str, runtime: ToolRuntime) -> Command:
    """Authenticate the user with the given email and password"""
    if email == runtime.context.email_address and password == runtime.context.password:
        return Command(update={
            "authenticated": True, 
            "messages": [ToolMessage(
                "Successfully authenticated", 
                tool_call_id=runtime.tool_call_id)]
        })
    else:
        return Command(update={
            "authenticated": False,
            "messages": [ToolMessage(
                "Authentication failed", 
                tool_call_id=runtime.tool_call_id)]
        })

### Implementation step 6

Imports the libraries and APIs needed by the following example: from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse; from typing import Callable. Defines a reusable Python function/tool. The docstring and type annotations communicate the tool purpose and input schema to the agent. Registers middleware that adapts model calls, prompts or tool availability at runtime. Keep the policy deterministic and test each branch.

**What to verify:** Check that the output matches the intended data flow, and inspect intermediate messages/tool calls when behavior is unclear.


In [ ]:
# Purpose: Imports the libraries and APIs needed by the following example: from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse; from typing import Callable. Defines a reusable Python function/tool. The docstring and type annotations communicate the tool purpose and input schema to the agent. Registers middleware that adapts model calls, prompts or tool availability at runtime. Keep the policy deterministic and test each branch.

from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from typing import Callable

@wrap_model_call
def dynamic_tool_call(request: ModelRequest, 
handler: Callable[[ModelRequest], ModelResponse]) -> ModelResponse:

    """Allow read inbox and send email tools only if user provides correct email and password"""

    authenticated = request.state.get("authenticated")
    
    if authenticated:
        tools = [check_inbox, send_email]
    else:
        tools = [authenticate]

    request = request.override(tools=tools) 
    return handler(request)

>Note: the prompts were modified since filming to constrain the model to more reliably match the filmed sequence. You may still experience different responses from the model, which is expected. You may need to modify the human message to provide appropriate responses.

### Implementation step 8

Imports the libraries and APIs needed by the following example: from langchain.agents.middleware import dynamic_prompt. Registers middleware that adapts model calls, prompts or tool availability at runtime. Keep the policy deterministic and test each branch.

**What to verify:** Check that the output matches the intended data flow, and inspect intermediate messages/tool calls when behavior is unclear.


In [ ]:
# Purpose: Imports the libraries and APIs needed by the following example: from langchain.agents.middleware import dynamic_prompt. Registers middleware that adapts model calls, prompts or tool availability at runtime. Keep the policy deterministic and test each branch.

from langchain.agents.middleware import dynamic_prompt

authenticated_prompt = """You are a helpful assistant that can check the inbox and send emails. 
Your first step after authentication is to check the inbox."""
unauthenticated_prompt = "You are a helpful assistant that can authenticate users."

@dynamic_prompt
def dynamic_prompt(request: ModelRequest) -> str:
    """Generate system prompt based on authentication status"""
    authenticated = request.state.get("authenticated")

    if authenticated:
        return authenticated_prompt
    else:
        return unauthenticated_prompt

### Implementation step 9

Imports the libraries and APIs needed by the following example: from langchain.agents import create_agent; from langgraph.checkpoint.memory import InMemorySaver; from langchain.agents.middleware import HumanInTheLoopMiddleware. Configures a chat model or agent. Model choice, system instructions and attached tools determine the capabilities available in this example. Enables checkpoint-backed short-term memory. Reuse the same thread identifier to continue a conversation; a new ID starts a separate thread. Demonstrates a pause/resume control path. Review the proposed action before resuming; approval should be explicit for consequential operations. Registers middleware that adapts model calls, prompts or tool availability at runtime. Keep the policy deterministic and test each branch.

**What to verify:** Check that the output matches the intended data flow, and inspect intermediate messages/tool calls when behavior is unclear.


In [ ]:
# Purpose: Imports the libraries and APIs needed by the following example: from langchain.agents import create_agent; from langgraph.checkpoint.memory import InMemorySaver; from langchain.agents.middleware import HumanInTheLoopMiddleware. Configures a chat model or agent. Model choice, system instructions and attached tools determine the capabilities available in this example. Enables checkpoint-backed short-term memory. Reuse the same thread identifier to continue a conversation; a new ID starts a separate thread. Demonstrates a pause/resume control path. Review the proposed action before resuming; approval should be explicit for consequential operations. Registers middleware that adapts model calls, prompts or tool availability at runtime. Keep the policy deterministic and test each branch.

from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import HumanInTheLoopMiddleware

agent = create_agent(
    "gpt-5-nano",
    tools=[authenticate, check_inbox, send_email],
    checkpointer=InMemorySaver(),
    state_schema=AuthenticatedState,
    context_schema=EmailContext,
    middleware=[
        dynamic_tool_call, 
        dynamic_prompt,
        HumanInTheLoopMiddleware(
            interrupt_on={
                "authenticate": False,
                "check_inbox": False,
                "send_email": True,
            })
        ]
    )


### Implementation step 10

Imports the libraries and APIs needed by the following example: from langchain.messages import HumanMessage. Executes the model/agent with an input payload. Inspect the returned object to understand messages, structured data, tool activity or state changes. Builds a typed conversation message. Message roles and ordering provide the context the model uses for its next response. Enables checkpoint-backed short-term memory. Reuse the same thread identifier to continue a conversation; a new ID starts a separate thread. Inspects or prints the result so you can verify the output structure and follow the execution path.

**What to verify:** Check that the output matches the intended data flow, and inspect intermediate messages/tool calls when behavior is unclear.


In [ ]:
# Purpose: Imports the libraries and APIs needed by the following example: from langchain.messages import HumanMessage. Executes the model/agent with an input payload. Inspect the returned object to understand messages, structured data, tool activity or state changes. Builds a typed conversation message. Message roles and ordering provide the context the model uses for its next response. Enables checkpoint-backed short-term memory. Reuse the same thread identifier to continue a conversation; a new ID starts a separate thread. Inspects or prints the result so you can verify the output structure and follow the execution path.

from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}

response = agent.invoke(
    {"messages": [HumanMessage(content="julie@example.com, password123")]},
    context=EmailContext(),
    config=config
)

print(response['messages'][-1].content)

### Implementation step 11

Executes the model/agent with an input payload. Inspect the returned object to understand messages, structured data, tool activity or state changes. Builds a typed conversation message. Message roles and ordering provide the context the model uses for its next response. Inspects or prints the result so you can verify the output structure and follow the execution path.

**What to verify:** Check that the output matches the intended data flow, and inspect intermediate messages/tool calls when behavior is unclear.


In [ ]:
# Purpose: Executes the model/agent with an input payload. Inspect the returned object to understand messages, structured data, tool activity or state changes. Builds a typed conversation message. Message roles and ordering provide the context the model uses for its next response. Inspects or prints the result so you can verify the output structure and follow the execution path.


response = agent.invoke(
    {"messages": [HumanMessage(content="any draft is fine. don't check back.")]},
    context=EmailContext(),
    config=config
)

print(response['messages'][-1].content)

### Implementation step 12

Demonstrates a pause/resume control path. Review the proposed action before resuming; approval should be explicit for consequential operations. Inspects or prints the result so you can verify the output structure and follow the execution path.

**What to verify:** Check that the output matches the intended data flow, and inspect intermediate messages/tool calls when behavior is unclear.


In [ ]:
# Purpose: Demonstrates a pause/resume control path. Review the proposed action before resuming; approval should be explicit for consequential operations. Inspects or prints the result so you can verify the output structure and follow the execution path.

print(response['__interrupt__'][0].value['action_requests'][0]['args']['body'])

### Implementation step 13

Imports the libraries and APIs needed by the following example: from langgraph.types import Command. Executes the model/agent with an input payload. Inspect the returned object to understand messages, structured data, tool activity or state changes. Demonstrates a pause/resume control path. Review the proposed action before resuming; approval should be explicit for consequential operations. Inspects or prints the result so you can verify the output structure and follow the execution path.

**What to verify:** Check that the output matches the intended data flow, and inspect intermediate messages/tool calls when behavior is unclear.


In [ ]:
# Purpose: Imports the libraries and APIs needed by the following example: from langgraph.types import Command. Executes the model/agent with an input payload. Inspect the returned object to understand messages, structured data, tool activity or state changes. Demonstrates a pause/resume control path. Review the proposed action before resuming; approval should be explicit for consequential operations. Inspects or prints the result so you can verify the output structure and follow the execution path.

from langgraph.types import Command

response = agent.invoke(
    Command( 
        resume={"decisions": [{"type": "approve"}]}  # or "reject"
    ), 
    config=config # Same thread ID to resume the paused conversation
)

print(response["messages"][-1].content)

### Implementation step 14

Imports the libraries and APIs needed by the following example: from pprint import pprint. Inspects or prints the result so you can verify the output structure and follow the execution path.

**What to verify:** Check that the output matches the intended data flow, and inspect intermediate messages/tool calls when behavior is unclear.


In [ ]:
# Purpose: Imports the libraries and APIs needed by the following example: from pprint import pprint. Inspects or prints the result so you can verify the output structure and follow the execution path.

from pprint import pprint

pprint(response)